# 🌍 Introduction to Geospatial Libraries — Exercise Notebook

---

> **How to use this notebook:**  
> Each section contains **worked examples** you can run, followed by **practice exercises**.  
> Read the examples carefully, then attempt the exercises in the solution cells.  
> Use `Shift + Enter` to run a cell.

---

## 📋 Table of Contents

| # | Library | Purpose |
|---|---------|----------|
| 0 | Setup & Environment | Google Colab, packages, data download |
| 1 | **Shapely** | Vector Geometry |
| 2 | **Fiona** | Vector I/O |
| 3 | **GeoPandas** | Vector Analysis |
| 4 | **Rasterio** | Raster I/O |
| 5 | **Xarray** | N-D Array Handling |
| 6 | **Rioxarray** | Raster/N-D Bridge |
| 7 | **scipy.stats** | Statistical Analysis |
| 8 | **Folium** | Interactive Mapping |
| 9 | **Leafmap** | Rapid Visualization |
| 10 | **Google Earth Engine (ee)** | Cloud GIS API |
| 11 | **rasterstats** | Zonal Statistics |
| 12 | **Matplotlib** | Static Visualization |
| 13 | **Plotly** | Interactive Visualization |

---

# 0️⃣ Setup & Environment

---

> **Google Colab** is a hosted Jupyter notebook environment. No local installation is required.  
> Every session runs on a fresh Ubuntu Linux machine in the cloud.

| Feature | Google Colab | Local Jupyter |
|---------|:---:|:---:|
| Installation needed | ❌ | ✅ |
| Free GPU/TPU | ✅ | ❌ |
| Internet required | ✅ | ❌ |
| Easy sharing | ✅ | ⚠️ |

---

### 💡 Example 0.1 — Hello World & Python Version

In [ ]:
import sys
print('Hello World')
print('Python version:', sys.version)
env = 'Google Colab' if 'google.colab' in sys.modules else 'Local Jupyter'
print('Running in:', env)

### 💡 Example 0.2 — Package Management

In [ ]:
import pandas as pd
import geopandas as gpd

# Install a package if needed (uncomment):
# !pip install rioxarray
# import rioxarray

# List all installed packages:
# !pip list -v

### 💡 Example 0.3 — Data & Output Folders

In [ ]:
import os

data_folder   = 'data'
output_folder = 'output'

for folder in [data_folder, output_folder]:
    if not os.path.exists(folder):
        os.mkdir(folder)
        print(f'Created folder: {folder}')
    else:
        print(f'Folder already exists: {folder}')

### 💡 Example 0.4 — Download Helper & Sample Data

In [ ]:
import requests

def download(url):
    """Download a file from url into the data folder."""
    filename = os.path.join(data_folder, os.path.basename(url))
    if not os.path.exists(filename):
        with requests.get(url, stream=True, allow_redirects=True) as r:
            with open(filename, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        print('Downloaded', filename)
    else:
        print('Already exists:', filename)

# Download tract-level shapefiles for three US states
base_url   = 'https://www2.census.gov/geo/tiger/TIGER2025/TRACT/'
zones_files = ['tl_2025_38_tract.zip', 'tl_2025_39_tract.zip', 'tl_2025_40_tract.zip']

for z in zones_files:
    download(base_url + z)

### ✏️ Exercise 0 — Practice Questions

**Q1.** Print your name and the current Python version in one print statement.

**Q2.** Create an additional folder called `'logs'` inside `data_folder` using `os.mkdir`.

**Q3.** Modify the `download()` function to print the file size in KB after downloading.

**Q4.** Download the county-level shapefile for North Dakota: `tl_2025_38_county.zip` from the same TIGER base URL (replace `TRACT` with `COUNTY` in the path).

**Q5.** Write a function `list_data_files()` that prints all files currently in the `data` folder.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
# Q1:
import sys
print(f'My name is Alex | Python {sys.version.split()[0]}')

# Q2:
logs_path = os.path.join(data_folder, 'logs')
os.makedirs(logs_path, exist_ok=True)

# Q3: add after f.write() loop:
size_kb = os.path.getsize(filename) / 1024
print(f'  Size: {size_kb:.1f} KB')

# Q4:
download('https://www2.census.gov/geo/tiger/TIGER2025/COUNTY/tl_2025_38_county.zip')

# Q5:
def list_data_files():
    for f in os.listdir(data_folder):
        print(f)
list_data_files()
```
</details>

---

# 1️⃣ Vector Geometry: Shapely

---

> **Shapely** creates and manipulates planar geometric objects (Points, Lines, Polygons) entirely in memory — no file I/O required.  
> It is the building block for almost every other Python geospatial library.

| Geometry Type | Constructor |
|---|---|
| Point | `Point(x, y)` |
| LineString | `LineString([(x1,y1), (x2,y2), ...])` |
| Polygon | `Polygon(shell, [holes])` |
| MultiPoint | `MultiPoint([...])` |
| MultiLineString | `MultiLineString([...])` |
| MultiPolygon | `MultiPolygon([...])` |
| GeometryCollection | `GeometryCollection([...])` |

---

### 💡 Example 1.1 — The Seven Geometry Types

In [ ]:
from shapely.geometry import (
    Point, LineString, Polygon,
    MultiPoint, MultiLineString, MultiPolygon,
    GeometryCollection
)

pt   = Point(5, 2)
ls   = LineString([(1,5),(4,4),(4,1),(2,2),(3,2)])
poly = Polygon([(1,5),(2,2),(4,1),(4,4),(1,5)])
mp   = MultiPoint([(5,2),(1,3),(3,4),(3,2)])
mls  = MultiLineString([[(1,5),(4,4),(4,1)],[(1,2),(2,4)]])
outer = [(1,5),(2,2),(4,1),(4,4),(1,5)]
hole  = [(0,2),(1,2),(1,3),(0,3),(0,2)]
mpoly = MultiPolygon([Polygon(outer, [hole])])
gc    = GeometryCollection([mp, ls])

for name, geom in [('POINT',pt),('LINESTRING',ls),('POLYGON',poly),
                   ('MULTIPOINT',mp),('MULTILINESTRING',mls),
                   ('MULTIPOLYGON',mpoly),('GEOMETRYCOLLECTION',gc)]:
    print(f'{name}: {geom.wkt[:60]}')
    display(geom)

### 💡 Example 1.2 — Spatial Predicates & Measurements

In [ ]:
from shapely.geometry import Point, LineString, Polygon
from shapely import wkt

# Buffer + intersection
pt     = Point(5, 5)
buffer = pt.buffer(3)
line   = LineString([(8,5),(12,5)])
print('Buffer intersects line:', buffer.intersects(line))
display(buffer)

# Polygon area and centroid
outer = [(0,0),(6,0),(6,6),(0,6),(0,0)]
hole  = [(2,2),(4,2),(4,4),(2,4),(2,2)]
poly  = Polygon(outer, [hole])
print('Area with hole:', poly.area)
print('Centroid:', poly.centroid)
display(poly)

# WKT round-trip
wkt_str  = poly.wkt
restored = wkt.loads(wkt_str)
print('WKT restored:', restored.geom_type)

### 💡 Example 1.3 — Convex Hull & Geometry Filtering

In [ ]:
from shapely.geometry import MultiPoint, GeometryCollection, Point, LineString

# Convex hull of a MultiPoint
mp   = MultiPoint([(1,1),(5,1),(3,4),(2,2),(4,3)])
hull = mp.convex_hull
print('Convex Hull:', hull.wkt)
display(GeometryCollection([mp, hull]))

# Extract only Points from a GeometryCollection
gc     = GeometryCollection([Point(1,1), LineString([(0,0),(2,2)]), Point(5,5), Point(3,4)])
points = [g for g in gc.geoms if g.geom_type == 'Point']
print(f'Extracted {len(points)} points from collection')
for p in points:
    print(' ', p)

### ✏️ Exercise 1 — Practice Questions

**Q1.** Create a `Point` at (3, 7) and a `Polygon` representing a 4×4 square centred on the origin. Check whether the point is **within** the polygon.

**Q2.** Create two overlapping `Polygon` objects. Compute and display their **intersection**, **union**, and **difference**.

**Q3.** Build a `LineString` with at least 5 coordinates. Print its **length** and the coordinates of the **midpoint** (use `interpolate(0.5, normalized=True)`).

**Q4.** Create a `MultiPolygon` of three non-overlapping squares. Compute the **total area** and the **bounding box** (`bounds`).

**Q5.** Given the WKT string `'LINESTRING (0 0, 1 1, 2 0, 3 1)'`, load it with `shapely.wkt.loads`, compute its **convex hull**, and print the hull's WKT.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
from shapely.geometry import Point, Polygon, MultiPolygon, LineString
from shapely import wkt as shapely_wkt

# Q1:
pt  = Point(3, 7)
sq  = Polygon([(-2,-2),(2,-2),(2,2),(-2,2)])
print('Within:', pt.within(sq))  # False

# Q2:
p1 = Polygon([(0,0),(4,0),(4,4),(0,4)])
p2 = Polygon([(2,2),(6,2),(6,6),(2,6)])
print('Intersection:', p1.intersection(p2).area)
print('Union area:', p1.union(p2).area)
print('Difference area:', p1.difference(p2).area)

# Q3:
ls  = LineString([(0,0),(1,2),(3,3),(5,1),(7,4)])
mid = ls.interpolate(0.5, normalized=True)
print('Length:', ls.length, '  Midpoint:', mid)

# Q4:
squares = MultiPolygon([Polygon([(i*5,0),(i*5+4,0),(i*5+4,4),(i*5,4)]) for i in range(3)])
print('Total area:', squares.area, '  Bounds:', squares.bounds)

# Q5:
ls2  = shapely_wkt.loads('LINESTRING (0 0, 1 1, 2 0, 3 1)')
hull = ls2.convex_hull
print('Hull WKT:', hull.wkt)
```
</details>

---

# 2️⃣ Vector I/O: Fiona

---

> **Fiona** reads and writes geospatial vector formats (Shapefiles, GeoJSON, GeoPackage) through a clean Pythonic interface to GDAL.  
> Every feature is returned as a Python dictionary — easy to inspect and transform.

| Task | Fiona method |
|------|--------------|
| Open for reading | `fiona.open(path, 'r')` |
| Get CRS | `src.crs` |
| Iterate features | `for feat in src:` |
| Open for writing | `fiona.open(path, 'w', driver=..., crs=..., schema=...)` |
| Write feature | `dst.write({...})` |

---

### 💡 Example 2.1 — Install & Read a Shapefile

In [ ]:
# !pip install fiona   # uncomment if needed
import fiona

file = r'data/tl_2025_38_tract.shp'

with fiona.open(file, 'r') as src:
    print('CRS       :', src.crs)
    print('Driver    :', src.driver)
    print('Schema    :', src.schema)
    print('Feature count:', len(src))
    # Print first feature
    first = next(iter(src))
    print('\nFirst geometry type:', first['geometry']['type'])
    print('First properties   :', dict(first['properties']))

### 💡 Example 2.2 — Write a New Shapefile

In [ ]:
import fiona
from fiona.crs import from_epsg

schema = {
    'geometry'  : 'Point',
    'properties': {'name': 'str:20', 'value': 'float'}
}

out_path = 'output/sample_points.shp'

with fiona.open(out_path, 'w', driver='ESRI Shapefile',
                crs=from_epsg(4326), schema=schema) as dst:
    dst.write({
        'geometry'  : {'type': 'Point', 'coordinates': (10, 20)},
        'properties': {'name': 'SiteA', 'value': 3.14}
    })
    dst.write({
        'geometry'  : {'type': 'Point', 'coordinates': (85.3, 27.7)},
        'properties': {'name': 'Kathmandu', 'value': 1400.0}
    })

print('Written:', out_path)

### ✏️ Exercise 2 — Practice Questions

**Q1.** Open `tl_2025_38_tract.shp` and print the **total feature count** and the **unique values** of the `COUNTYFP` property.

**Q2.** Read all features into a Python list of dicts and print the properties of the **last** feature.

**Q3.** Write a new GeoJSON file (`output/cities.geojson`) containing **three** city `Point` features with `name` and `population` properties.

**Q4.** Open the shapefile you wrote in Example 2.2 and print every feature's name and coordinates.

**Q5.** Copy all features where `ALAND` > 5×10⁸ (large tracts) into a new shapefile `output/large_tracts.shp`, keeping the same schema and CRS.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import fiona
from fiona.crs import from_epsg
file = r'data/tl_2025_38_tract.shp'

# Q1:
with fiona.open(file) as src:
    print('Count:', len(src))
    counties = {f['properties']['COUNTYFP'] for f in src}
    print('Unique COUNTYFP:', sorted(counties))

# Q2:
with fiona.open(file) as src:
    feats = list(src)
print('Last feature props:', dict(feats[-1]['properties']))

# Q3:
schema3 = {'geometry':'Point','properties':{'name':'str:30','population':'int'}}
with fiona.open('output/cities.geojson','w',driver='GeoJSON',
                crs=from_epsg(4326),schema=schema3) as dst:
    for nm,pop,lon,lat in [('Fargo',120000,-96.79,46.88),
                            ('Bismarck',74000,-100.78,46.80),
                            ('Grand Forks',59000,-97.03,47.92)]:
        dst.write({'geometry':{'type':'Point','coordinates':(lon,lat)},
                   'properties':{'name':nm,'population':pop}})

# Q4:
with fiona.open('output/sample_points.shp') as src:
    for f in src:
        print(f['properties']['name'], f['geometry']['coordinates'])

# Q5:
with fiona.open(file) as src:
    schema5 = src.schema.copy()
    crs5    = src.crs
    large   = [f for f in src if f['properties']['ALAND'] > 5e8]
with fiona.open('output/large_tracts.shp','w',driver='ESRI Shapefile',
                crs=crs5,schema=schema5) as dst:
    for f in large: dst.write(f)
print(f'Wrote {len(large)} large tracts')
```
</details>

---

# 3️⃣ Vector Analysis: GeoPandas

---

> **GeoPandas** extends Pandas DataFrames with a `geometry` column, enabling vectorized spatial operations directly on tabular data.

| Operation | Method |
|-----------|--------|
| Read file | `gpd.read_file(path)` |
| Calculate area | `gdf.area` |
| Reproject | `gdf.to_crs(epsg=...)` |
| Spatial join | `gpd.sjoin(left, right, predicate=...)` |
| Dissolve | `gdf.dissolve(by='column')` |
| Buffer | `gdf.buffer(distance)` |
| Write file | `gdf.to_file(path)` |

---

### 💡 Example 3.1 — Create & Inspect a GeoDataFrame

In [ ]:
import geopandas as gpd
from shapely.geometry import Polygon

data = {
    'city'      : ['Fargo', 'Bismarck', 'Grand Forks'],
    'population': [160000, 43000, 59000],
    'geometry'  : [
        Polygon([(2,5),(7,9),(6,8),(5,1)]),
        Polygon([(10,10),(10,12),(12,12),(12,10)]),
        Polygon([(0,0),(3,0),(3,3),(0,3)])
    ]
}
gdf = gpd.GeoDataFrame(data, crs='EPSG:4326')
gdf['area'] = gdf.area
print(gdf[['city','population','area']])
gdf.plot(column='population', legend=True, figsize=(6,4))

### 💡 Example 3.2 — Read Shapefile, Filter & Reproject

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

nd    = gpd.read_file(r'data/tl_2025_38_tract.zip')
print('CRS:', nd.crs)
print('Shape:', nd.shape)

# Filter to a single county
county = nd[nd['COUNTYFP'] == '017']
print(f'\nCounty 017 tracts: {len(county)}')

# Reproject to a projected CRS for accurate area calculation
county_proj = county.to_crs(epsg=32614)
county_proj['area_km2'] = county_proj.area / 1e6
print(county_proj[['GEOID','area_km2']].head())

county.plot(color='lightyellow', edgecolor='black', figsize=(6,5))
plt.title('County 017 Tracts')
plt.show()

### 💡 Example 3.3 — Spatial Join

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd

# Polygons (zones)
zones = gpd.GeoDataFrame({
    'zone': ['A','B'],
    'geometry': [
        Polygon([(0,0),(5,0),(5,5),(0,5)]),
        Polygon([(5,0),(10,0),(10,5),(5,5)])
    ]
}, crs='EPSG:4326')

# Points (locations)
pts = gpd.GeoDataFrame({
    'name': ['P1','P2','P3'],
    'geometry': [Point(2,2), Point(7,3), Point(4,4)]
}, crs='EPSG:4326')

joined = gpd.sjoin(pts, zones, how='left', predicate='within')
print(joined[['name','zone']])

### ✏️ Exercise 3 — Practice Questions

**Q1.** Read `tl_2025_38_tract.zip` into a GeoDataFrame. Print the CRS, total feature count, and column names.

**Q2.** Reproject the GeoDataFrame to EPSG:32614 (UTM zone 14N), compute area in km², and print the **top 5 largest tracts** by area.

**Q3.** Filter the data to tracts in county `COUNTYFP == '035'`. Dissolve all tracts into a single county polygon and plot it.

**Q4.** Create a GeoDataFrame of 5 random points inside North Dakota's bounding box. Spatial-join them to the tract layer to find which tract each point falls in.

**Q5.** Buffer all tract geometries by 500 m (first reproject to a metric CRS), then save the result to `output/tracts_buffered.shp`.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import geopandas as gpd, numpy as np
from shapely.geometry import Point

nd = gpd.read_file(r'data/tl_2025_38_tract.zip')

# Q1:
print(nd.crs, len(nd), nd.columns.tolist())

# Q2:
nd_proj = nd.to_crs(epsg=32614)
nd_proj['area_km2'] = nd_proj.area / 1e6
print(nd_proj.nlargest(5,'area_km2')[['GEOID','area_km2']])

# Q3:
c035 = nd[nd['COUNTYFP']=='035'].dissolve()
c035.plot(); import matplotlib.pyplot as plt; plt.title('County 035'); plt.show()

# Q4:
minx,miny,maxx,maxy = nd.total_bounds
rng = np.random.default_rng(42)
rand_pts = gpd.GeoDataFrame(
    geometry=[Point(rng.uniform(minx,maxx), rng.uniform(miny,maxy)) for _ in range(5)],
    crs=nd.crs)
joined = gpd.sjoin(rand_pts, nd[['GEOID','geometry']], how='left', predicate='within')
print(joined[['geometry','GEOID']])

# Q5:
nd_m = nd.to_crs(epsg=32614)
nd_m['geometry'] = nd_m.buffer(500)
nd_m.to_file('output/tracts_buffered.shp')
print('Saved buffered tracts')
```
</details>

---

# 4️⃣ Raster I/O: Rasterio

---

> **Rasterio** reads and writes GeoTIFFs and other raster formats, exposing data as NumPy arrays with full geospatial metadata.

| Task | Code |
|------|------|
| Open raster | `rasterio.open(path)` |
| Read band | `src.read(1)` |
| CRS & transform | `src.crs`, `src.transform` |
| Clip to geometry | `rasterio.mask.mask(src, shapes)` |
| Write raster | `rasterio.open(path,'w', **profile)` |

---

### 💡 Example 4.1 — Read Raster Metadata

In [ ]:
# !pip install rasterio   # uncomment if needed
import rasterio
import numpy as np

# --- Using a real file: uncomment and replace path ---
# with rasterio.open('data/my_dem.tif') as src:
#     print('CRS     :', src.crs)
#     print('Bounds  :', src.bounds)
#     print('Shape   :', src.height, src.width)
#     print('Bands   :', src.count)
#     band1 = src.read(1)
#     print('Min/Max :', band1.min(), band1.max())

# Mock demo (no file required)
class MockRaster:
    crs     = 'EPSG:4326'
    bounds  = (10.0, 20.0, 30.0, 40.0)
    count   = 4
    height  = 512
    width   = 512
    def read(self, band): return np.random.randint(0,255,(self.height,self.width))

src = MockRaster()
print(f'CRS   : {src.crs}')
print(f'Bands : {src.count}')
print(f'Size  : {src.height} x {src.width}')
band = src.read(1)
print(f'Mean pixel value: {band.mean():.2f}')

### 💡 Example 4.2 — Clipping a Raster with a Vector Mask

In [ ]:
# Conceptual example — requires real raster file
# from rasterio.mask import mask as rio_mask
# import geopandas as gpd
#
# gdf  = gpd.read_file('data/my_boundary.shp').to_crs('EPSG:4326')
# shapes = [geom.__geo_interface__ for geom in gdf.geometry]
#
# with rasterio.open('data/my_dem.tif') as src:
#     clipped, clipped_transform = rio_mask(src, shapes, crop=True)
#     meta = src.meta.copy()
#
# meta.update({'height': clipped.shape[1], 'width': clipped.shape[2],
#              'transform': clipped_transform})
#
# with rasterio.open('output/clipped.tif', 'w', **meta) as dst:
#     dst.write(clipped)
#
print('Clipping concept: rasterio.mask.mask(src, shapes, crop=True)')
print('Result is a NumPy array + new affine transform.')

### ✏️ Exercise 4 — Practice Questions

**Q1.** What function and parameters would you use to open a GeoTIFF in **read mode** and print its CRS, number of bands, and bounding box?

**Q2.** Using the mock raster in Example 4.1, simulate reading bands 1–4 and print the **mean**, **min**, and **max** of each band.

**Q3.** What is the purpose of the `affine transform` stored in a rasterio dataset? Write pseudocode to convert a pixel (row=100, col=200) to geographic coordinates.

**Q4.** Write the code outline for saving a NumPy array as a new single-band GeoTIFF using `rasterio.open` in write mode (define a `profile` dict).

**Q5.** What Rasterio function would you call to clip a raster to a polygon boundary? List the required input arguments.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import rasterio, numpy as np

# Q1:
# with rasterio.open('data/dem.tif') as src:
#     print(src.crs, src.count, src.bounds)

# Q2 (mock):
class MockRaster:
    count=4; height=256; width=256
    def read(self,b): return np.random.randint(0,255,(self.height,self.width))
src = MockRaster()
for b in range(1, src.count+1):
    arr = src.read(b)
    print(f'Band {b}: mean={arr.mean():.1f} min={arr.min()} max={arr.max()}')

# Q3 pseudocode:
# x = transform.c + col * transform.a
# y = transform.f + row * transform.e

# Q4:
# profile = {'driver':'GTiff','dtype':'float32','width':256,'height':256,
#            'count':1,'crs':'EPSG:4326','transform':my_transform}
# with rasterio.open('output/new.tif','w',**profile) as dst:
#     dst.write(my_array, 1)

# Q5:
# from rasterio.mask import mask
# clipped, transform = mask(src, shapes, crop=True)
# Required: src (open dataset), shapes (list of geometries), crop=True
```
</details>

---

# 5️⃣ N-D Array Handling: Xarray

---

> **Xarray** adds named dimensions and coordinates to NumPy arrays — ideal for multi-dimensional data like climate grids and time series.

| Concept | Description |
|---------|-------------|
| `DataArray` | Labelled N-D array |
| `Dataset` | Dict of DataArrays sharing coordinates |
| `.sel()` | Select by label value |
| `.isel()` | Select by integer index |
| `.mean(dim=...)` | Reduce along a dimension |

---

### 💡 Example 5.1 — Create & Select from a DataArray

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

times = pd.date_range('2023-01-01', periods=5)
lats  = np.arange(40, 45)

temp  = xr.DataArray(
    np.random.rand(5, 5) * 30,
    coords=[times, lats],
    dims=['time', 'lat'],
    name='temperature'
)

print(temp)
print('\nAt lat=42:', temp.sel(lat=42).values)
print('Mean over time:', temp.mean(dim='time').values)

### 💡 Example 5.2 — Dataset & Arithmetic

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

times = pd.date_range('2023-01-01', periods=5)
lats  = np.arange(40, 45)

ds = xr.Dataset({
    'temperature': xr.DataArray(np.random.rand(5,5)*30, dims=['time','lat'],
                                coords={'time':times,'lat':lats}),
    'humidity'   : xr.DataArray(np.random.rand(5,5)*100, dims=['time','lat'],
                                coords={'time':times,'lat':lats})
})

print(ds)
print('\nMean temperature:', ds['temperature'].mean().item(), '°C')
print('Max humidity    :', ds['humidity'].max().item(), '%')

### ✏️ Exercise 5 — Practice Questions

**Q1.** Create a 3-D `DataArray` with dimensions `(time=4, lat=5, lon=5)` filled with random values. Print its shape and the mean along the `time` dimension.

**Q2.** Using `.sel()`, extract a time slice for `'2023-01-03'` from the DataArray in Example 5.1 and print the resulting values.

**Q3.** Create a `Dataset` with two variables (`'ndvi'` and `'evi'`) sharing the same `(time, lat, lon)` coordinates. Compute the difference `ndvi - evi`.

**Q4.** Use `.resample(time='M').mean()` on a daily temperature DataArray (30 days) to calculate **monthly means**. Print the result.

**Q5.** Save your Dataset to a NetCDF file using `.to_netcdf('output/my_dataset.nc')` and reload it with `xr.open_dataset()`.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import xarray as xr, numpy as np, pandas as pd

# Q1:
times = pd.date_range('2023-01-01',periods=4)
lats  = np.arange(10,15); lons = np.arange(80,85)
da = xr.DataArray(np.random.rand(4,5,5),dims=['time','lat','lon'],
                  coords={'time':times,'lat':lats,'lon':lons})
print(da.shape, da.mean(dim='time'))

# Q2:
times2 = pd.date_range('2023-01-01',periods=5)
lats2  = np.arange(40,45)
temp   = xr.DataArray(np.random.rand(5,5)*30,dims=['time','lat'],
                      coords={'time':times2,'lat':lats2})
print(temp.sel(time='2023-01-03').values)

# Q3:
ds = xr.Dataset({'ndvi': da, 'evi': xr.DataArray(np.random.rand(4,5,5),
                 dims=['time','lat','lon'],coords=da.coords)})
diff = ds['ndvi'] - ds['evi']
print(diff)

# Q4:
daily = xr.DataArray(np.random.rand(30),
    dims=['time'], coords={'time': pd.date_range('2023-01-01',periods=30)})
print(daily.resample(time='ME').mean())

# Q5:
ds.to_netcdf('output/my_dataset.nc')
ds2 = xr.open_dataset('output/my_dataset.nc')
print(ds2)
```
</details>

---

# 6️⃣ Raster/N-D Bridge: Rioxarray

---

> **Rioxarray** extends Xarray with a `.rio` accessor that exposes CRS, spatial transforms, reprojection, and clipping — bridging Rasterio and Xarray seamlessly.

| `.rio` method | Purpose |
|---------------|---------|
| `.crs` | Get the CRS |
| `.write_crs(epsg)` | Set / write the CRS |
| `.reproject('EPSG:...')` | Reproject |
| `.clip(geometries, crs)` | Clip to vector boundary |
| `.to_raster(path)` | Save to GeoTIFF |

---

### 💡 Example 6.1 — Access CRS via .rio Accessor

In [ ]:
# !pip install rioxarray   # uncomment if needed
import xarray as xr
import numpy as np
import rioxarray   # registers the .rio accessor

# Create a mock spatial DataArray
da = xr.DataArray(
    np.random.rand(4, 4),
    dims=('y', 'x'),
    coords={'y': [40.5, 41.5, 42.5, 43.5], 'x': [-97.5, -96.5, -95.5, -94.5]}
)
da = da.rio.write_crs('EPSG:4326')
print('CRS  :', da.rio.crs)
print('Shape:', da.rio.shape)
print('Bounds:', da.rio.bounds())

### ✏️ Exercise 6 — Practice Questions

**Q1.** Create a 5×5 spatial DataArray with lat/lon coordinates. Assign it EPSG:4326 using `.rio.write_crs()` and print its CRS and spatial bounds.

**Q2.** What method would you use to **reproject** a rioxarray DataArray from EPSG:4326 to EPSG:32614? Write the one-line call.

**Q3.** Describe the workflow (in comments/pseudocode) to **clip** a rioxarray DataArray to a Shapely polygon boundary.

**Q4.** What is the difference between `rasterio.open()` and `rioxarray.open_rasterio()`? When would you prefer each?

**Q5.** Write code to save a rioxarray DataArray to a GeoTIFF file at `output/result.tif`.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import xarray as xr, numpy as np, rioxarray

# Q1:
da = xr.DataArray(np.random.rand(5,5),dims=('y','x'),
    coords={'y':[40,41,42,43,44],'x':[-97,-96,-95,-94,-93]})
da = da.rio.write_crs('EPSG:4326')
print(da.rio.crs, da.rio.bounds())

# Q2:
# da_projected = da.rio.reproject('EPSG:32614')

# Q3:
# from shapely.geometry import mapping
# clipped = da.rio.clip([mapping(my_polygon)], crs='EPSG:4326', drop=True)

# Q4:
# rasterio.open() → low-level, band-by-band NumPy access
# rioxarray.open_rasterio() → returns labelled Xarray DataArray with .rio
# Prefer rioxarray when chaining spatial ops; rasterio for low-level control

# Q5:
# da.rio.to_raster('output/result.tif')
```
</details>

---

# 7️⃣ Statistical Analysis: scipy.stats

---

> **scipy.stats** provides probability distributions, hypothesis tests, and descriptive statistics critical for spatial data analysis.

| Function | Purpose |
|----------|---------|
| `stats.describe(data)` | Summary statistics |
| `stats.ttest_1samp(data, popmean)` | One-sample t-test |
| `stats.pearsonr(x, y)` | Pearson correlation |
| `stats.norm.fit(data)` | Fit normal distribution |
| `stats.kstest(data, 'norm')` | Normality test |

---

### 💡 Example 7.1 — Descriptive Stats & Hypothesis Test

In [ ]:
from scipy import stats
import numpy as np

# Simulated elevation data (metres)
elevation = np.array([125.5, 130.2, 120.8, 145.0, 118.9, 132.1, 128.7, 140.3])

desc = stats.describe(elevation)
print(f'N     : {desc.nobs}')
print(f'Mean  : {desc.mean:.2f} m')
print(f'Std   : {np.sqrt(desc.variance):.2f} m')
print(f'Min   : {desc.minmax[0]:.2f} m')
print(f'Max   : {desc.minmax[1]:.2f} m')

# One-sample t-test: is our mean significantly different from 135 m?
t_stat, p_val = stats.ttest_1samp(elevation, 135.0)
print(f'\nt-statistic : {t_stat:.4f}')
print(f'p-value     : {p_val:.4f}')
print('Significant difference (p<0.05):', p_val < 0.05)

### 💡 Example 7.2 — Correlation & Distribution Fitting

In [ ]:
from scipy import stats
import numpy as np

# Simulate NDVI and rainfall
np.random.seed(42)
rainfall = np.random.normal(500, 80, 30)
ndvi     = 0.003 * rainfall + np.random.normal(0, 0.05, 30)

r, p = stats.pearsonr(rainfall, ndvi)
print(f'Pearson r = {r:.4f},  p = {p:.4f}')

# Fit a normal distribution to rainfall
mu, sigma = stats.norm.fit(rainfall)
print(f'\nFitted Normal: mean={mu:.1f}, std={sigma:.1f}')

### ✏️ Exercise 7 — Practice Questions

**Q1.** Generate 50 random elevation values (normal, mean=200, std=25). Compute and print the full descriptive statistics using `stats.describe()`.

**Q2.** Perform a **one-sample t-test** to check if the mean of your elevation data is significantly different from 210 m (α = 0.05). State your conclusion.

**Q3.** Simulate two land-cover NDVI arrays (e.g., forest vs grassland). Use `stats.ttest_ind()` to test if their means are significantly different.

**Q4.** Compute the **Pearson correlation** between two spatial variables of your choice (e.g., altitude vs temperature). Interpret the result.

**Q5.** Use `stats.kstest()` to test whether a dataset follows a normal distribution. Print the statistic and p-value and state your conclusion.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
from scipy import stats; import numpy as np; np.random.seed(0)

# Q1:
elev = np.random.normal(200, 25, 50)
d = stats.describe(elev)
print(d)

# Q2:
t, p = stats.ttest_1samp(elev, 210)
print(f't={t:.3f}, p={p:.4f}', 'significant' if p<0.05 else 'not significant')

# Q3:
forest   = np.random.normal(0.7, 0.05, 30)
grass    = np.random.normal(0.5, 0.07, 30)
t2, p2   = stats.ttest_ind(forest, grass)
print(f't={t2:.3f}, p={p2:.4f}', 'different' if p2<0.05 else 'same')

# Q4:
alt  = np.random.uniform(500, 5000, 40)
temp = -0.006 * alt + 25 + np.random.normal(0, 1, 40)
r, p3 = stats.pearsonr(alt, temp)
print(f'r={r:.3f} — strong negative correlation' if r < -0.7 else f'r={r:.3f}')

# Q5:
data = np.random.normal(0,1,100)
ks, p4 = stats.kstest(data,'norm')
print(f'KS={ks:.4f}, p={p4:.4f}', 'normal' if p4>0.05 else 'not normal')
```
</details>

---

# 8️⃣ Interactive Mapping: Folium

---

> **Folium** generates interactive Leaflet.js maps in Python notebooks and HTML files — no JavaScript required.

| Element | Code |
|---------|------|
| Base map | `folium.Map(location, zoom_start, tiles)` |
| Marker | `folium.Marker(location, popup, tooltip)` |
| Circle | `folium.Circle(location, radius, color)` |
| GeoJSON layer | `folium.GeoJson(data).add_to(m)` |
| Save map | `m.save('map.html')` |

---

### 💡 Example 8.1 — Basic Folium Map with Markers

In [ ]:
import folium

fargo_coords = (46.8772, -96.7898)

m = folium.Map(location=fargo_coords, zoom_start=10, tiles='OpenStreetMap')

folium.Marker(fargo_coords,
              popup='<b>Fargo, ND</b>',
              tooltip='Click me').add_to(m)

folium.Circle(fargo_coords, radius=5000,
              color='blue', fill=True, fill_opacity=0.2,
              tooltip='5 km radius').add_to(m)

# Display in notebook
m

### 💡 Example 8.2 — Multiple Markers & Save to HTML

In [ ]:
import folium

cities = [
    ('Fargo',       46.8772, -96.7898, 'blue'),
    ('Bismarck',    46.8083, -100.7837, 'red'),
    ('Grand Forks', 47.9253, -97.0329,  'green'),
]

m = folium.Map(location=[47.0, -99.0], zoom_start=6)

for name, lat, lon, color in cities:
    folium.Marker(
        [lat, lon],
        popup=name,
        icon=folium.Icon(color=color)
    ).add_to(m)

m.save('output/nd_cities_map.html')
print('Map saved to output/nd_cities_map.html')
m

### ✏️ Exercise 8 — Practice Questions

**Q1.** Create a Folium map centred on **Kathmandu, Nepal** (27.7172° N, 85.3240° E) with zoom level 12 and `CartoDB positron` tiles.

**Q2.** Add **three markers** for landmarks in any city you choose. Each marker should have a custom popup with the landmark name and a tooltip.

**Q3.** Draw a `folium.Polygon` around a rectangular area of your choice and fill it with a semi-transparent colour.

**Q4.** Add a `folium.GeoJson` layer to the map using the GeoDataFrame from Section 3 (convert with `gdf.to_json()`).

**Q5.** Save the final map to `output/my_folium_map.html` and confirm the file size with `os.path.getsize()`.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import folium, os

# Q1:
m = folium.Map(location=[27.7172,85.3240], zoom_start=12, tiles='CartoDB positron')
m

# Q2:
landmarks = [('Pashupatinath',27.7105,85.3487),
             ('Boudhanath',27.7215,85.3620),
             ('Swayambhunath',27.7149,85.2904)]
for nm,lat,lon in landmarks:
    folium.Marker([lat,lon],popup=nm,tooltip=nm).add_to(m)

# Q3:
folium.Polygon(locations=[[27.68,85.28],[27.68,85.38],[27.76,85.38],[27.76,85.28]],
              color='green',fill=True,fill_opacity=0.2).add_to(m)

# Q4:
# import geopandas as gpd
# gdf = gpd.read_file(r'data/tl_2025_38_tract.zip')
# folium.GeoJson(gdf.to_json(), name='Tracts').add_to(m)

# Q5:
m.save('output/my_folium_map.html')
print(os.path.getsize('output/my_folium_map.html'), 'bytes')
```
</details>

---

# 9️⃣ Rapid Visualization: Leafmap

---

> **Leafmap** simplifies interactive geospatial visualization — add basemaps, vector layers, and rasters in a few lines.

| Task | Method |
|------|--------|
| Create map | `leafmap.Map(center, zoom)` |
| Add basemap | `m.add_basemap('OpenStreetMap')` |
| Add GeoDataFrame | `m.add_gdf(gdf, layer_name)` |
| Add GeoJSON URL | `m.add_geojson(url, layer_name)` |
| Save to HTML | `m.to_html(path)` |
| Zoom to data | `m.zoom_to_gdf(gdf)` |

---

### 💡 Example 9.1 — Leafmap Map with Vector Layer

In [ ]:
# !pip install leafmap   # uncomment if needed
import leafmap
import geopandas as gpd

m = leafmap.Map(center=(46.88, -99.0), zoom=6)
m.add_basemap('OpenStreetMap')

# Load and display ND tract data
gdf = gpd.read_file(r'data/tl_2025_38_tract.zip')
m.add_gdf(gdf, layer_name='ND Tracts',
          style={'color': 'navy', 'weight': 0.5, 'fillOpacity': 0.2})
m.zoom_to_gdf(gdf)

m

### 💡 Example 9.2 — Save to HTML

In [ ]:
output_path = r'output/nd_tracts_leafmap.html'
m.to_html(output_path)
print('Saved to:', output_path)

### ✏️ Exercise 9 — Practice Questions

**Q1.** Create a Leafmap map centred on Kathmandu, Nepal. Add both `'OpenStreetMap'` and `'Esri.WorldImagery'` basemaps.

**Q2.** Load `tl_2025_39_tract.zip` (Ohio) with GeoPandas and add it to a Leafmap map with a **yellow fill** and **red edges**.

**Q3.** Add a `folium.Marker` to the Leafmap map (hint: access `m.to_html()` or use `m.add_marker(location=...)`).

**Q4.** Filter the ND tract layer to a single county, add it as a **separate named layer**, and enable the layer control.

**Q5.** Save the complete multi-layer map to `output/multi_layer_map.html`.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import leafmap, geopandas as gpd

# Q1:
m = leafmap.Map(center=[27.7172,85.3240], zoom=12)
m.add_basemap('OpenStreetMap')
m.add_basemap('Esri.WorldImagery')

# Q2:
oh = gpd.read_file(r'data/tl_2025_39_tract.zip')
m2 = leafmap.Map()
m2.add_gdf(oh, layer_name='Ohio Tracts',
           style={'color':'red','weight':0.5,'fillColor':'yellow','fillOpacity':0.4})
m2.zoom_to_gdf(oh)

# Q3:
m.add_marker(location=[27.7172,85.3240], popup='Kathmandu')

# Q4:
nd  = gpd.read_file(r'data/tl_2025_38_tract.zip')
c11 = nd[nd['COUNTYFP']=='011']
m3  = leafmap.Map()
m3.add_gdf(nd,  layer_name='All ND Tracts')
m3.add_gdf(c11, layer_name='County 011', style={'color':'red','weight':1})

# Q5:
m3.to_html('output/multi_layer_map.html')
print('Saved')
```
</details>

---

# 🔟 Cloud GIS API: Google Earth Engine (ee)

---

> **Google Earth Engine** enables petabyte-scale geospatial analysis on Google's cloud infrastructure.  
> Computation is **server-side** — only results are returned to your local machine.

| Step | Code |
|------|------|
| Authenticate | `ee.Authenticate()` |
| Initialize | `ee.Initialize(project='your-project')` |
| Load collection | `ee.ImageCollection('LANDSAT/...')` |
| Filter dates | `.filterDate('YYYY-MM-DD', 'YYYY-MM-DD')` |
| Compute | `.median()`, `.mean()`, `.mosaic()` |
| Get value | `.getInfo()` |

---

### 💡 Example 10.1 — Authenticate, Initialize & Load Landsat 8

In [ ]:
import ee

# Authenticate (only needed once per session)
ee.Authenticate()
ee.Initialize(project='rsclass01')  # replace with your project ID

# Load Landsat 8 TOA collection and filter by date
collection = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA')
    .filterDate('2024-01-01', '2024-06-30')
)

print('Images in collection:', collection.size().getInfo())

# Compute median composite
median_img = collection.median()
print('Median image band names:', median_img.bandNames().getInfo())

### ✏️ Exercise 10 — Practice Questions

**Q1.** After `ee.Initialize()`, load the **Sentinel-2 SR** collection (`COPERNICUS/S2_SR`) and print the number of images available for 2024.

**Q2.** Filter the Landsat 8 collection to images that intersect **North Dakota's bounding box** (use `filterBounds()`).

**Q3.** Compute the **median NDVI** of a Landsat 8 collection over one month. NDVI = (NIR − Red) / (NIR + Red), where NIR = B5, Red = B4.

**Q4.** Is Earth Engine computation performed **locally** or on **Google's servers**? Explain the lazy evaluation model and what `.getInfo()` triggers.

**Q5.** What function would you call to **export** an Earth Engine image to Google Drive? List the required parameters.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
# Q1:
s2 = ee.ImageCollection('COPERNICUS/S2_SR').filterDate('2024-01-01','2024-12-31')
print(s2.size().getInfo())

# Q2:
nd_bbox = ee.Geometry.Rectangle([-104.05, 45.93, -96.55, 49.00])
nd_col  = ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA').filterBounds(nd_bbox)
print(nd_col.size().getInfo())

# Q3:
def add_ndvi(img):
    ndvi = img.normalizedDifference(['B5','B4']).rename('NDVI')
    return img.addBands(ndvi)
med_ndvi = nd_col.map(add_ndvi).select('NDVI').median()
print(med_ndvi.bandNames().getInfo())

# Q4:
# Computation is server-side on Google's infrastructure.
# GEE uses lazy evaluation — no computation until .getInfo() or Export is called.

# Q5:
# ee.batch.Export.image.toDrive(
#   image=med_ndvi, description='NDVI_export',
#   scale=30, region=nd_bbox, fileFormat='GeoTIFF'
# ).start()
```
</details>

---

# 1️⃣1️⃣ Zonal Statistics: rasterstats

---

> **rasterstats** calculates summary statistics (mean, max, sum, …) of raster pixels that fall within vector boundaries — the core of zonal statistics.

| Parameter | Description |
|-----------|-------------|
| `vector` | Path or GeoDataFrame of zones |
| `raster` | Path or NumPy array |
| `stats` | List: `['mean','max','min','count','sum']` |
| `affine` | Affine transform (when passing array) |
| Returns | List of dicts, one per feature |

---

### 💡 Example 11.1 — Zonal Statistics on a Mock Raster

In [ ]:
# !pip install rasterstats   # uncomment if needed
import numpy as np
from shapely.geometry import Polygon
from rasterstats import zonal_stats
from affine import Affine

# 5×5 mock raster
raster = np.array([
    [10, 20, 30, 40, 50],
    [15, 25, 35, 45, 55],
    [12, 22, 32, 42, 52],
    [18, 28, 38, 48, 58],
    [11, 21, 31, 41, 51]
], dtype=float)

# Affine(xres, xskew, xorigin, yskew, yres, yorigin)
affine = Affine(1.0, 0.0, 0.0,
                0.0,-1.0, 5.0)

# Define two zones
zone1 = Polygon([(0,0),(0,3),(2,3),(2,0)])
zone2 = Polygon([(2,0),(2,3),(5,3),(5,0)])

for i, zone in enumerate([zone1, zone2], 1):
    result = zonal_stats(zone, raster, affine=affine,
                         stats=['mean','max','min','sum','count'],
                         nodata=-9999)
    print(f'Zone {i}: {result[0]}')

### ✏️ Exercise 11 — Practice Questions

**Q1.** Modify the mock raster in Example 11.1 to include a **NoData value of -9999** in two cells. Re-run zonal stats and confirm the `count` reflects only valid pixels.

**Q2.** Add `'std'` and `'median'` to the `stats` list. Print the updated results for both zones.

**Q3.** Create a 10×10 raster representing a DEM (elevation 0–500 m). Define 4 non-overlapping zones and compute `mean`, `max`, `min` for each.

**Q4.** What does the `affine` transform parameter represent? How does it relate the raster pixel grid to real-world coordinates?

**Q5.** (Conceptual) If you had a real GeoTIFF DEM and a shapefile of watersheds, write the pseudocode/code outline to compute **mean elevation per watershed** using `zonal_stats`.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import numpy as np
from shapely.geometry import Polygon
from rasterstats import zonal_stats
from affine import Affine

# Q1:
raster = np.array([[10,20,30,40,50],[15,-9999,35,45,55],
                   [12,22,32,-9999,52],[18,28,38,48,58],[11,21,31,41,51]],dtype=float)
affine = Affine(1.0,0.0,0.0,0.0,-1.0,5.0)
zone1  = Polygon([(0,0),(0,3),(2,3),(2,0)])
print(zonal_stats(zone1,raster,affine=affine,stats=['mean','count'],nodata=-9999))

# Q2:
zone2 = Polygon([(2,0),(2,3),(5,3),(5,0)])
for z in [zone1,zone2]:
    print(zonal_stats(z,raster,affine=affine,
                      stats=['mean','max','min','std','median'],nodata=-9999))

# Q3:
dem    = np.random.randint(0,501,(10,10)).astype(float)
aff10  = Affine(1.0,0.0,0.0,0.0,-1.0,10.0)
zones3 = [Polygon([(0,0),(0,5),(5,5),(5,0)]),
          Polygon([(5,0),(5,5),(10,5),(10,0)]),
          Polygon([(0,5),(0,10),(5,10),(5,5)]),
          Polygon([(5,5),(5,10),(10,10),(10,5)])]
for i,z in enumerate(zones3,1):
    print(f'Zone {i}:',zonal_stats(z,dem,affine=aff10,stats=['mean','max','min']))

# Q4:
# Affine(xres, xskew, xUL, yskew, yres, yUL)
# Maps pixel (col, row) → (x = xUL + col*xres, y = yUL + row*yres)

# Q5:
# stats = zonal_stats('watersheds.shp', 'dem.tif',
#                      stats=['mean'], geojson_out=True)
# for feat in stats: print(feat['properties']['mean'])
```
</details>

---

# 1️⃣2️⃣ Static Visualization: Matplotlib

---

> **Matplotlib** is Python's foundational plotting library — highly customisable for maps, charts, and multi-panel figures.

| Plot type | Function |
|-----------|----------|
| Line chart | `plt.plot(x, y)` |
| Histogram | `plt.hist(data)` |
| Scatter | `plt.scatter(x, y)` |
| Geo plot | `gdf.plot(ax=ax, column=..., cmap=...)` |
| Save figure | `plt.savefig('file.png', dpi=150)` |

---

### 💡 Example 12.1 — Basic Line & Scatter Plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(0, 10, 100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Line plot
ax1.plot(x, np.cos(x), label='cos(x)', color='blue')
ax1.plot(x, np.sin(x), label='sin(x)', color='orange')
ax1.set_title('Trig Functions'); ax1.legend(); ax1.grid(True)

# Scatter plot
np.random.seed(0)
ax2.scatter(np.random.rand(50)*10, np.random.rand(50)*10,
            c=np.random.rand(50), cmap='viridis', s=80)
ax2.set_title('Random Scatter')

plt.tight_layout()
plt.show()

### 💡 Example 12.2 — Choropleth Map of ND Tracts

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

nd      = gpd.read_file(r'data/tl_2025_38_tract.zip')
nd_proj = nd.to_crs(epsg=32614)
nd_proj['area_km2'] = nd_proj.area / 1e6

fig, ax = plt.subplots(figsize=(10, 8))
nd_proj.plot(ax=ax, column='area_km2', cmap='YlOrRd',
             legend=True, legend_kwds={'label': 'Area (km²)'})
ax.set_title('North Dakota Census Tracts — Area', fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.savefig('output/nd_choropleth.png', dpi=150)
plt.show()
print('Saved: output/nd_choropleth.png')

### ✏️ Exercise 12 — Practice Questions

**Q1.** Create a 2×2 grid of subplots showing: (a) line plot of a sine wave, (b) histogram of 500 random normal values, (c) scatter of two correlated variables, (d) bar chart of 5 categories.

**Q2.** Load `tl_2025_38_tract.zip` and create a **choropleth map** coloured by `ALAND` (land area) using the `'Blues'` colormap. Add a title and a legend.

**Q3.** Plot North Dakota tracts coloured by county (`COUNTYFP`) using a categorical colormap. Add a legend showing the county codes.

**Q4.** Create a **side-by-side map** comparing two different counties (e.g., 005 and 017) using `plt.subplots(1, 2)`. Use the same color scheme for both.

**Q5.** Save a high-resolution version (300 DPI) of one of your maps to `output/hires_map.png` and confirm the file was written.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import matplotlib.pyplot as plt, numpy as np, geopandas as gpd, os

# Q1:
fig, axes = plt.subplots(2,2,figsize=(10,8))
x = np.linspace(0,2*np.pi,100)
axes[0,0].plot(x,np.sin(x)); axes[0,0].set_title('Sine')
axes[0,1].hist(np.random.randn(500),bins=25); axes[0,1].set_title('Histogram')
xc = np.random.rand(50); axes[1,0].scatter(xc,xc+np.random.rand(50)*0.3); axes[1,0].set_title('Scatter')
axes[1,1].bar(['A','B','C','D','E'],[5,3,8,2,7]); axes[1,1].set_title('Bar')
plt.tight_layout(); plt.show()

# Q2:
nd = gpd.read_file(r'data/tl_2025_38_tract.zip')
fig,ax = plt.subplots(figsize=(10,8))
nd.plot(ax=ax,column='ALAND',cmap='Blues',legend=True); ax.set_title('ALAND'); plt.show()

# Q3:
fig,ax = plt.subplots(figsize=(10,8))
nd.plot(ax=ax,column='COUNTYFP',cmap='tab20',legend=True,legend_kwds={'ncol':3})
ax.set_title('ND Tracts by County'); ax.set_axis_off(); plt.show()

# Q4:
c005 = nd[nd['COUNTYFP']=='005']; c017 = nd[nd['COUNTYFP']=='017']
fig,axes = plt.subplots(1,2,figsize=(12,5))
for ax,gdf,nm in zip(axes,[c005,c017],['County 005','County 017']):
    gdf.plot(ax=ax,color='lightblue',edgecolor='k'); ax.set_title(nm); ax.set_axis_off()
plt.tight_layout(); plt.show()

# Q5:
fig,ax = plt.subplots(figsize=(10,8))
nd.plot(ax=ax,color='lightgreen',edgecolor='grey',linewidth=0.3)
ax.set_title('ND Tracts'); ax.set_axis_off()
plt.savefig('output/hires_map.png',dpi=300)
print('File size:', os.path.getsize('output/hires_map.png'), 'bytes')
```
</details>

---

# 1️⃣3️⃣ Interactive Visualization: Plotly

---

> **Plotly** creates publication-quality interactive charts — hover, zoom, and filter in the browser.

| Chart type | Function |
|------------|----------|
| Line | `px.line(df, x=, y=)` |
| Scatter | `px.scatter(df, x=, y=, color=)` |
| Bar | `px.bar(df, x=, y=)` |
| Choropleth | `px.choropleth(df, geojson=, locations=, color=)` |
| 3D scatter | `px.scatter_3d(df, x=, y=, z=, color=)` |

---

### 💡 Example 13.1 — Interactive Scatter Plot

In [ ]:
import plotly.express as px
import pandas as pd

df = pd.DataFrame({
    'Year'  : [2000, 2005, 2010, 2015, 2020, 2000, 2005, 2010, 2015, 2020],
    'NDVI'  : [0.55, 0.60, 0.58, 0.65, 0.70, 0.40, 0.42, 0.38, 0.45, 0.50],
    'Region': ['Forest']*5 + ['Grassland']*5,
    'Area'  : [100, 110, 105, 120, 130, 80, 82, 79, 88, 95]
})

fig = px.scatter(
    df, x='Year', y='NDVI', color='Region', size='Area',
    title='NDVI Trend by Land Cover Region',
    hover_name='Region'
)
fig.show()

### 💡 Example 13.2 — Interactive Line & Bar Charts

In [ ]:
import plotly.express as px
import pandas as pd

# Line chart — temperature trend
df_temp = pd.DataFrame({
    'Month': ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'],
    'Temp_C': [-13, -10, -3, 7, 15, 21, 24, 23, 16, 7, -3, -11]
})
fig_line = px.line(df_temp, x='Month', y='Temp_C',
                   title='Average Monthly Temperature — Fargo, ND',
                   markers=True)
fig_line.show()

# Bar chart — population
df_pop = pd.DataFrame({
    'City': ['Fargo','Bismarck','Grand Forks','Minot','Dickinson'],
    'Pop' : [125000, 74000, 59000, 48000, 25000]
})
fig_bar = px.bar(df_pop, x='City', y='Pop',
                 title='North Dakota City Populations',
                 color='City')
fig_bar.show()

### ✏️ Exercise 13 — Practice Questions

**Q1.** Create a Plotly **line chart** showing monthly precipitation for a city of your choice (make up 12 monthly values). Add axis labels and a title.

**Q2.** Create a **grouped bar chart** comparing NDVI values for three land-cover types across four seasons using `px.bar(barmode='group')`.

**Q3.** Create a **3D scatter plot** (`px.scatter_3d`) with axes representing Longitude, Latitude, and Elevation for 20 random points.

**Q4.** Create a **choropleth map** of US states using `px.choropleth` with a dummy variable (e.g., random population values). Use the `'usa-states'` location mode.

**Q5.** Export one of your Plotly figures to a **standalone HTML file** using `fig.write_html('output/my_plotly_chart.html')` and confirm it was saved.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import plotly.express as px, pandas as pd, numpy as np, os

# Q1:
df1 = pd.DataFrame({'Month':list(range(1,13)),
                    'Precip_mm':[30,25,35,50,70,80,60,55,45,40,35,28]})
px.line(df1,x='Month',y='Precip_mm',title='Monthly Precipitation',markers=True).show()

# Q2:
df2 = pd.DataFrame({'Season':['Spring','Summer','Autumn','Winter']*3,
                    'LandCover':['Forest']*4+['Grassland']*4+['Cropland']*4,
                    'NDVI':[0.6,0.8,0.5,0.3,0.4,0.6,0.3,0.2,0.5,0.7,0.4,0.1]})
px.bar(df2,x='Season',y='NDVI',color='LandCover',barmode='group',
       title='NDVI by Season and Land Cover').show()

# Q3:
np.random.seed(1)
df3 = pd.DataFrame({'Lon':np.random.uniform(-100,-96,20),
                    'Lat':np.random.uniform(46,49,20),
                    'Elev':np.random.uniform(300,600,20)})
px.scatter_3d(df3,x='Lon',y='Lat',z='Elev',color='Elev',
              title='3D Point Cloud').show()

# Q4:
states = ['CA','TX','FL','NY','PA','IL','OH','GA','NC','MI']
df4 = pd.DataFrame({'state':states,'value':np.random.randint(1e6,40e6,len(states))})
px.choropleth(df4,locations='state',locationmode='USA-states',color='value',
              scope='usa',title='Random State Values').show()

# Q5:
fig5 = px.scatter(df3,x='Lon',y='Lat',color='Elev',title='ND Points')
fig5.write_html('output/my_plotly_chart.html')
print(os.path.getsize('output/my_plotly_chart.html'),'bytes')
```
</details>

---

---

# 🎉 Congratulations — All 14 Sections Complete!

---

| ✅ | Section | Library |
|----|---------|----------|
| ✅ | 0 | Setup & Environment |
| ✅ | 1 | Shapely — Vector Geometry |
| ✅ | 2 | Fiona — Vector I/O |
| ✅ | 3 | GeoPandas — Vector Analysis |
| ✅ | 4 | Rasterio — Raster I/O |
| ✅ | 5 | Xarray — N-D Arrays |
| ✅ | 6 | Rioxarray — Raster/N-D Bridge |
| ✅ | 7 | scipy.stats — Statistics |
| ✅ | 8 | Folium — Interactive Maps |
| ✅ | 9 | Leafmap — Rapid Visualization |
| ✅ | 10 | Google Earth Engine |
| ✅ | 11 | rasterstats — Zonal Statistics |
| ✅ | 12 | Matplotlib — Static Visualization |
| ✅ | 13 | Plotly — Interactive Visualization |

---

## 🚀 Next Steps

| Domain | Libraries |
|--------|-----------|
| 🛰️ Remote Sensing | `rasterio`, `rioxarray`, `earthpy`, `ee` |
| 🗺️ Web Mapping | `folium`, `leafmap`, `ipyleaflet`, `kepler.gl` |
| 📊 Spatial Stats | `pysal`, `esda`, `libpysal` |
| ☁️ Cloud GIS | `geemap`, `planetary-computer`, `stac` |
| 🔧 Automation | `snakemake`, `prefect`, `airflow` |

---

### Happy Mapping! 🌍💻

*Every dataset has a story — geospatial libraries help you read it.*

---